In [1]:
from dataclasses import dataclass
from pathlib import Path
import io
import re

import zstandard as zstd

In [2]:
@dataclass
class PgnGame:
  game_index: int
  tags: dict
  movetext: str
  raw_pgn: str

In [3]:
class PgnZstParser:
  def __init__(self, path):
    self.path = Path(path)

  def parse_first_n(self, n_games):
    games = []

    for game in self.iter_games():
      games.append(game)

      if len(games) >= n_games:
        break

    return games

  def parse_first_n_with_eval(self, n_games):
    games = []

    for game in self.iter_games_with_eval():
      games.append(game)

      if len(games) >= n_games:
        break

    return games

  def iter_games(self):
    game_index = 0

    for game_lines in self._iter_raw_game_lines():
      yield self._build_game(game_index, game_lines)
      game_index += 1

  def iter_games_with_eval(self):
    game_index = 0
    eval_game_index = 0

    for game_lines in self._iter_raw_game_lines():
      raw_pgn = "\n".join(game_lines).strip()

      if "%eval" not in raw_pgn:
        game_index += 1
        continue

      yield self._build_game(
        game_index=game_index,
        game_lines=game_lines,
      )

      game_index += 1
      eval_game_index += 1

  def _iter_raw_game_lines(self):
    game_lines = []
    seen_movetext = False

    with open(self.path, "rb") as file:
      dctx = zstd.ZstdDecompressor()

      with dctx.stream_reader(file) as byte_stream:
        text_stream = io.TextIOWrapper(
          byte_stream,
          encoding="utf-8",
          errors="replace",
        )

        for line in text_stream:
          line = line.rstrip("\n")

          if self._is_new_game_start(
            line=line,
            game_lines=game_lines,
            seen_movetext=seen_movetext,
          ):
            yield game_lines

            game_lines = []
            seen_movetext = False

          game_lines.append(line)

          if self._is_movetext_line(line):
            seen_movetext = True

        if game_lines:
          yield game_lines

  def _is_new_game_start(self, line, game_lines, seen_movetext):
    if not game_lines:
      return False

    if not seen_movetext:
      return False

    return line.startswith("[Event ")

  def _is_movetext_line(self, line):
    if not line.strip():
      return False

    if line.startswith("[") and line.endswith("]"):
      return False

    return True

  def _build_game(self, game_index, game_lines):
    raw_pgn = "\n".join(game_lines).strip()
    tags = {}
    movetext_lines = []
    in_movetext = False

    for line in game_lines:
      if self._is_tag_line(line) and not in_movetext:
        key, value = self._parse_tag_line(line)
        tags[key] = value
        continue

      if line.strip():
        in_movetext = True

      if in_movetext:
        movetext_lines.append(line)

    movetext = "\n".join(movetext_lines).strip()

    return PgnGame(
      game_index=game_index,
      tags=tags,
      movetext=movetext,
      raw_pgn=raw_pgn,
    )

  def _is_tag_line(self, line):
    return bool(re.match(r'^\[[A-Za-z0-9_]+ ".*"\]$', line))

  def _parse_tag_line(self, line):
    match = re.match(r'^\[([A-Za-z0-9_]+) "(.*)"\]$', line)

    if match is None:
      raise ValueError(f"Could not parse tag line: {line}")

    return match.group(1), match.group(2)

In [4]:
pgn_path = "../data/raw/lichess_db_standard_rated_2017-05.pgn.zst"

parser = PgnZstParser(pgn_path)
eval_games = parser.parse_first_n_with_eval(10)

In [5]:
len(eval_games)

10

In [6]:
eval_games[0].tags

{'Event': 'Rated Blitz tournament https://lichess.org/tournament/HmYBgXC1',
 'Site': 'https://lichess.org/ufqUImzs',
 'White': 'Lexman661',
 'Black': 'aspekt',
 'Result': '0-1',
 'UTCDate': '2017.04.30',
 'UTCTime': '22:00:01',
 'WhiteElo': '1950',
 'BlackElo': '1952',
 'WhiteRatingDiff': '-11',
 'BlackRatingDiff': '+12',
 'ECO': 'A01',
 'Opening': 'Nimzo-Larsen Attack: Modern Variation #2',
 'TimeControl': '300+0',
 'Termination': 'Normal'}

In [7]:
print(eval_games[0].movetext)

1. b3 { [%eval 0.03] [%clk 0:05:00] } 1... e5 { [%eval -0.01] [%clk 0:05:00] } 2. Bb2 { [%eval -0.09] [%clk 0:04:58] } 2... d6 { [%eval 0.19] [%clk 0:05:00] } 3. g3 { [%eval 0.11] [%clk 0:04:56] } 3... Nf6 { [%eval 0.13] [%clk 0:04:58] } 4. Bg2 { [%eval -0.13] [%clk 0:04:55] } 4... g6?! { [%eval 0.37] [%clk 0:04:57] } 5. d3 { [%eval 0.06] [%clk 0:04:36] } 5... Bg7 { [%eval 0.05] [%clk 0:04:56] } 6. c4 { [%eval -0.02] [%clk 0:04:34] } 6... O-O { [%eval -0.04] [%clk 0:04:55] } 7. e3 { [%eval -0.16] [%clk 0:04:33] } 7... Re8 { [%eval 0.02] [%clk 0:04:46] } 8. Ne2 { [%eval -0.09] [%clk 0:04:32] } 8... d5 { [%eval 0.03] [%clk 0:04:40] } 9. O-O { [%eval -0.05] [%clk 0:04:27] } 9... Nc6 { [%eval -0.08] [%clk 0:04:37] } 10. Nbc3 { [%eval 0.03] [%clk 0:04:23] } 10... Be6 { [%eval 0.21] [%clk 0:04:35] } 11. Rc1 { [%eval 0.12] [%clk 0:04:18] } 11... Qd7 { [%eval 0.47] [%clk 0:04:34] } 12. Re1?! { [%eval -0.27] [%clk 0:04:13] } 12... Rad8 { [%eval 0.13] [%clk 0:04:32] } 13. cxd5 { [%eval 0.05] [%c

In [8]:
all("%eval" in game.raw_pgn for game in eval_games)

True

In [9]:
import pandas as pd

def games_to_dataframe(games):
  rows = []

  for game in games:
    row = {
      "game_index": game.game_index,
      "movetext": game.movetext,
      "raw_pgn": game.raw_pgn,
      "has_eval": "%eval" in game.raw_pgn,
    }

    for key, value in game.tags.items():
      row[key] = value

    rows.append(row)

  return pd.DataFrame(rows)

In [10]:
df_eval_games = games_to_dataframe(eval_games)
df_eval_games.head()

,game_index,movetext,raw_pgn,has_eval,Event,Site,White,Black,Result,UTCDate,UTCTime,WhiteElo,BlackElo,WhiteRatingDiff,BlackRatingDiff,ECO,Opening,TimeControl,Termination
0,85,1. b3 { [%eval 0.03] [%clk 0:05:00] } 1... e5 ...,"[Event ""Rated Blitz tournament https://lichess...",True,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/ufqUImzs,Lexman661,aspekt,0-1,2017.04.30,22:00:01,1950,1952,-11,+12,A01,Nimzo-Larsen Attack: Modern Variation #2,300+0,Normal
1,90,1. e4 { [%eval 0.21] [%clk 0:05:00] } 1... d5 ...,"[Event ""Rated Blitz tournament https://lichess...",True,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/0xW79X0B,dalila77,ghotir,1/2-1/2,2017.04.30,22:00:01,1738,1733,+0,+0,B01,Scandinavian Defense: Mieses-Kotroc Variation,300+0,Normal
2,130,1. e4 { [%eval 0.35] [%clk 0:01:00] } 1... c6 ...,"[Event ""Rated Bullet tournament https://liches...",True,Rated Bullet tournament https://lichess.org/to...,https://lichess.org/avftQQN9,coloviczoki,robrml,1-0,2017.04.30,22:00:01,2136,2001,+8,-7,B12,"Caro-Kann Defense: Advance Variation, Botvinni...",60+0,Normal
3,144,1. Nf3 { [%eval 0.19] [%clk 0:05:00] } 1... Nc...,"[Event ""Rated Classical game""]\n[Site ""https:/...",True,Rated Classical game,https://lichess.org/wGdv6qxi,TheYams,amir188,1-0,2017.04.30,22:00:09,1607,1470,+7,-8,A04,Zukertort Opening: Black Mustang Defense,300+10,Time forfeit
4,151,1. e4 { [%eval 0.2] [%clk 0:10:00] } 1... c5 {...,"[Event ""Rated Classical tournament https://lic...",True,Rated Classical tournament https://lichess.org...,https://lichess.org/kbkdiaqQ,arrami01,adondevamos,1-0,2017.04.30,22:00:01,2061,2076,+13,-13,B84,"Sicilian Defense: Scheveningen Variation, Clas...",600+0,Time forfeit


## Audit data

In [11]:
import re

import pandas as pd

In [12]:
EVAL_PATTERN = re.compile(r"\[%eval\s+([^\]]+)\]")
CLOCK_PATTERN = re.compile(r"\[%clk\s+([^\]]+)\]")
MOVE_NUMBER_PATTERN = re.compile(r"\b\d+\.(?:\.\.)?")


def estimate_n_plies(movetext):
  text = re.sub(r"\{[^}]*\}", " ", movetext)

  tokens = text.split()
  result_tokens = {"1-0", "0-1", "1/2-1/2", "*"}

  move_tokens = []

  for token in tokens:
    if token in result_tokens:
      continue

    if re.match(r"^\d+\.(?:\.\.)?$", token):
      continue

    if token.startswith("$"):
      continue

    move_tokens.append(token)

  return len(move_tokens)


def audit_one_game(game):
  raw_pgn = game.raw_pgn
  movetext = game.movetext
  tags = game.tags

  eval_matches = EVAL_PATTERN.findall(raw_pgn)
  clock_matches = CLOCK_PATTERN.findall(raw_pgn)
  move_numbers = MOVE_NUMBER_PATTERN.findall(movetext)

  has_mate_eval = any(
    eval_value.strip().startswith("#")
    for eval_value in eval_matches
  )

  row = {
    "game_index": game.game_index,
    "Event": tags.get("Event"),
    "Site": tags.get("Site"),
    "White": tags.get("White"),
    "Black": tags.get("Black"),
    "Result": tags.get("Result"),
    "WhiteElo": tags.get("WhiteElo"),
    "BlackElo": tags.get("BlackElo"),
    "ECO": tags.get("ECO"),
    "Opening": tags.get("Opening"),
    "TimeControl": tags.get("TimeControl"),
    "Termination": tags.get("Termination"),
    "n_chars_raw_pgn": len(raw_pgn),
    "n_chars_movetext": len(movetext),
    "n_eval_annotations": len(eval_matches),
    "n_clock_annotations": len(clock_matches),
    "has_eval": len(eval_matches) > 0,
    "has_clock": len(clock_matches) > 0,
    "has_mate_eval": has_mate_eval,
    "first_eval_raw": eval_matches[0] if eval_matches else None,
    "last_eval_raw": eval_matches[-1] if eval_matches else None,
    "n_move_numbers_approx": len(move_numbers),
    "n_plies_approx": estimate_n_plies(movetext),
  }

  return row


def audit_games(games):
  rows = []

  for game in games:
    rows.append(audit_one_game(game))

  return pd.DataFrame(rows)

In [13]:
df_audit = audit_games(eval_games)
df_audit.head()

,game_index,Event,Site,White,Black,Result,WhiteElo,BlackElo,ECO,Opening,...,n_chars_movetext,n_eval_annotations,n_clock_annotations,has_eval,has_clock,has_mate_eval,first_eval_raw,last_eval_raw,n_move_numbers_approx,n_plies_approx
0,85,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/ufqUImzs,Lexman661,aspekt,0-1,1950,1952,A01,Nimzo-Larsen Attack: Modern Variation #2,...,2597,62,62,True,True,False,0.03,-5.27,124,62
1,90,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/0xW79X0B,dalila77,ghotir,1/2-1/2,1738,1733,B01,Scandinavian Defense: Mieses-Kotroc Variation,...,5759,140,140,True,True,False,0.21,0.0,280,140
2,130,Rated Bullet tournament https://lichess.org/to...,https://lichess.org/avftQQN9,coloviczoki,robrml,1-0,2136,2001,B12,"Caro-Kann Defense: Advance Variation, Botvinni...",...,2945,70,71,True,True,True,0.35,#1,138,71
3,144,Rated Classical game,https://lichess.org/wGdv6qxi,TheYams,amir188,1-0,1607,1470,A04,Zukertort Opening: Black Mustang Defense,...,531,13,13,True,True,False,0.19,-0.85,26,13
4,151,Rated Classical tournament https://lichess.org...,https://lichess.org/kbkdiaqQ,arrami01,adondevamos,1-0,2061,2076,B84,"Sicilian Defense: Scheveningen Variation, Clas...",...,5413,131,131,True,True,True,0.2,#8,258,131


In [14]:
df_audit.shape

(10, 23)

In [15]:
df_audit["has_eval"].value_counts(dropna=False)

has_eval
True    10
Name: count, dtype: int64

In [16]:
df_audit[
  [
    "game_index",
    "White",
    "Black",
    "Result",
    "WhiteElo",
    "BlackElo",
    "n_eval_annotations",
    "n_clock_annotations",
    "has_mate_eval",
  ]
].head(10)

,game_index,White,Black,Result,WhiteElo,BlackElo,n_eval_annotations,n_clock_annotations,has_mate_eval
0,85,Lexman661,aspekt,0-1,1950,1952,62,62,False
1,90,dalila77,ghotir,1/2-1/2,1738,1733,140,140,False
2,130,coloviczoki,robrml,1-0,2136,2001,70,71,True
3,144,TheYams,amir188,1-0,1607,1470,13,13,False
4,151,arrami01,adondevamos,1-0,2061,2076,131,131,True
5,184,jarquincastillo,angelasturias,1-0,1723,1702,67,67,False
6,200,ThaJarney,chefbrandolono,1-0,1075,1163,58,59,True
7,209,Ygreek,roguemarvel,0-1,2030,1955,34,34,True
8,219,denis889,jrhodes,1-0,1277,1199,41,41,False
9,224,laserany,Theache,1-0,1462,1455,37,37,False


In [17]:
summary_cols = [
  "n_chars_raw_pgn",
  "n_chars_movetext",
  "n_eval_annotations",
  "n_clock_annotations",
  "n_move_numbers_approx",
  "n_plies_approx",
]

df_audit[summary_cols].describe()

,n_chars_raw_pgn,n_chars_movetext,n_eval_annotations,n_clock_annotations,n_move_numbers_approx,n_plies_approx
count,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000
mean,3098.800000,2716.100000,65.300000,65.500000,129.500000,65.500000
std,1709.507908,1684.403122,40.949969,40.945085,81.536972,40.945085
min,895.000000,531.000000,13.000000,13.000000,26.000000,13.000000
25%,1918.750000,1575.000000,38.000000,38.000000,76.000000,38.000000
50%,2909.000000,2529.000000,60.000000,60.500000,119.500000,60.500000
75%,3318.000000,2906.250000,69.250000,70.000000,137.000000,70.000000
max,6167.000000,5759.000000,140.000000,140.000000,280.000000,140.000000


In [18]:
df_audit["evals_per_ply_approx"] = (
  df_audit["n_eval_annotations"] / df_audit["n_plies_approx"]
)

df_audit[
  [
    "game_index",
    "n_eval_annotations",
    "n_plies_approx",
    "evals_per_ply_approx",
  ]
].head(10)

,game_index,n_eval_annotations,n_plies_approx,evals_per_ply_approx
0,85,62,62,1.000000
1,90,140,140,1.000000
2,130,70,71,0.985915
3,144,13,13,1.000000
4,151,131,131,1.000000
5,184,67,67,1.000000
6,200,58,59,0.983051
7,209,34,34,1.000000
8,219,41,41,1.000000
9,224,37,37,1.000000


In [19]:
df_audit["evals_per_ply_approx"].describe()

count    10.000000
mean      0.996897
std       0.006577
min       0.983051
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000
Name: evals_per_ply_approx, dtype: float64

In [20]:
all_eval_values = []

for game in eval_games:
  all_eval_values.extend(EVAL_PATTERN.findall(game.raw_pgn))

df_eval_values = pd.DataFrame({
  "eval_raw": all_eval_values,
})

df_eval_values.head(20)

,eval_raw
0,0.03
1,-0.01
2,-0.09
3,0.19
4,0.11
5,0.13
6,-0.13
7,0.37
8,0.06
9,0.05


In [21]:
df_eval_values["eval_raw"].value_counts().head(20)

eval_raw
0.0      54
-0.08     7
0.1       7
-0.09     5
0.19      5
0.48      5
0.26      5
0.29      5
0.03      4
0.06      4
-0.04     4
0.54      4
0.52      4
0.72      4
2.5       4
0.08      4
0.18      4
0.14      4
2.47      4
0.37      3
Name: count, dtype: int64

In [22]:
df_eval_values[
  df_eval_values["eval_raw"].str.startswith("#", na=False)
].head(20)

,eval_raw
269,#2
270,#1
271,#1
410,#20
412,#17
414,#9
415,#8
512,#5
540,#1
571,#-3


In [36]:
df_eval_values.max()

eval_raw    9.98
dtype: str

In [24]:
df_audit.sort_values("n_eval_annotations").head(10)

,game_index,Event,Site,White,Black,Result,WhiteElo,BlackElo,ECO,Opening,...,n_eval_annotations,n_clock_annotations,has_eval,has_clock,has_mate_eval,first_eval_raw,last_eval_raw,n_move_numbers_approx,n_plies_approx,evals_per_ply_approx
3,144,Rated Classical game,https://lichess.org/wGdv6qxi,TheYams,amir188,1-0,1607,1470,A04,Zukertort Opening: Black Mustang Defense,...,13,13,True,True,False,0.19,-0.85,26,13,1.000000
7,209,Rated Classical game,https://lichess.org/5ZUG3wEi,Ygreek,roguemarvel,0-1,2030,1955,B28,"Sicilian Defense: O'Kelly Variation, Normal Sy...",...,34,34,True,True,True,0.14,#-1,64,34,1.000000
9,224,Rated Classical game,https://lichess.org/thHVpOWb,laserany,Theache,1-0,1462,1455,C61,Ruy Lopez: Bird Variation,...,37,37,True,True,False,0.18,14.53,74,37,1.000000
8,219,Rated Bullet game,https://lichess.org/EZB9tDuE,denis889,jrhodes,1-0,1277,1199,B01,Scandinavian Defense,...,41,41,True,True,False,0.33,-2.31,82,41,1.000000
6,200,Rated Blitz game,https://lichess.org/Qwv4PuNW,ThaJarney,chefbrandolono,1-0,1075,1163,A04,Zukertort Opening: Dutch Variation,...,58,59,True,True,True,0.18,#1,115,59,0.983051
0,85,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/ufqUImzs,Lexman661,aspekt,0-1,1950,1952,A01,Nimzo-Larsen Attack: Modern Variation #2,...,62,62,True,True,False,0.03,-5.27,124,62,1.000000
5,184,Rated Classical game,https://lichess.org/OcHTndW9,jarquincastillo,angelasturias,1-0,1723,1702,C34,"King's Gambit Accepted, Fischer Defense",...,67,67,True,True,False,0.3,1.79,134,67,1.000000
2,130,Rated Bullet tournament https://lichess.org/to...,https://lichess.org/avftQQN9,coloviczoki,robrml,1-0,2136,2001,B12,"Caro-Kann Defense: Advance Variation, Botvinni...",...,70,71,True,True,True,0.35,#1,138,71,0.985915
4,151,Rated Classical tournament https://lichess.org...,https://lichess.org/kbkdiaqQ,arrami01,adondevamos,1-0,2061,2076,B84,"Sicilian Defense: Scheveningen Variation, Clas...",...,131,131,True,True,True,0.2,#8,258,131,1.000000
1,90,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/0xW79X0B,dalila77,ghotir,1/2-1/2,1738,1733,B01,Scandinavian Defense: Mieses-Kotroc Variation,...,140,140,True,True,False,0.21,0.0,280,140,1.000000


In [25]:
df_audit.sort_values("evals_per_ply_approx").head(10)

,game_index,Event,Site,White,Black,Result,WhiteElo,BlackElo,ECO,Opening,...,n_eval_annotations,n_clock_annotations,has_eval,has_clock,has_mate_eval,first_eval_raw,last_eval_raw,n_move_numbers_approx,n_plies_approx,evals_per_ply_approx
6,200,Rated Blitz game,https://lichess.org/Qwv4PuNW,ThaJarney,chefbrandolono,1-0,1075,1163,A04,Zukertort Opening: Dutch Variation,...,58,59,True,True,True,0.18,#1,115,59,0.983051
2,130,Rated Bullet tournament https://lichess.org/to...,https://lichess.org/avftQQN9,coloviczoki,robrml,1-0,2136,2001,B12,"Caro-Kann Defense: Advance Variation, Botvinni...",...,70,71,True,True,True,0.35,#1,138,71,0.985915
0,85,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/ufqUImzs,Lexman661,aspekt,0-1,1950,1952,A01,Nimzo-Larsen Attack: Modern Variation #2,...,62,62,True,True,False,0.03,-5.27,124,62,1.000000
1,90,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/0xW79X0B,dalila77,ghotir,1/2-1/2,1738,1733,B01,Scandinavian Defense: Mieses-Kotroc Variation,...,140,140,True,True,False,0.21,0.0,280,140,1.000000
3,144,Rated Classical game,https://lichess.org/wGdv6qxi,TheYams,amir188,1-0,1607,1470,A04,Zukertort Opening: Black Mustang Defense,...,13,13,True,True,False,0.19,-0.85,26,13,1.000000
4,151,Rated Classical tournament https://lichess.org...,https://lichess.org/kbkdiaqQ,arrami01,adondevamos,1-0,2061,2076,B84,"Sicilian Defense: Scheveningen Variation, Clas...",...,131,131,True,True,True,0.2,#8,258,131,1.000000
5,184,Rated Classical game,https://lichess.org/OcHTndW9,jarquincastillo,angelasturias,1-0,1723,1702,C34,"King's Gambit Accepted, Fischer Defense",...,67,67,True,True,False,0.3,1.79,134,67,1.000000
7,209,Rated Classical game,https://lichess.org/5ZUG3wEi,Ygreek,roguemarvel,0-1,2030,1955,B28,"Sicilian Defense: O'Kelly Variation, Normal Sy...",...,34,34,True,True,True,0.14,#-1,64,34,1.000000
8,219,Rated Bullet game,https://lichess.org/EZB9tDuE,denis889,jrhodes,1-0,1277,1199,B01,Scandinavian Defense,...,41,41,True,True,False,0.33,-2.31,82,41,1.000000
9,224,Rated Classical game,https://lichess.org/thHVpOWb,laserany,Theache,1-0,1462,1455,C61,Ruy Lopez: Bird Variation,...,37,37,True,True,False,0.18,14.53,74,37,1.000000


In [26]:
idx = df_audit.sort_values("n_eval_annotations").iloc[0]["game_index"]

game = next(
  game for game in eval_games
  if game.game_index == idx
)

print(game.raw_pgn)

[Event "Rated Classical game"]
[Site "https://lichess.org/wGdv6qxi"]
[White "TheYams"]
[Black "amir188"]
[Result "1-0"]
[UTCDate "2017.04.30"]
[UTCTime "22:00:09"]
[WhiteElo "1607"]
[BlackElo "1470"]
[WhiteRatingDiff "+7"]
[BlackRatingDiff "-8"]
[ECO "A04"]
[Opening "Zukertort Opening: Black Mustang Defense"]
[TimeControl "300+10"]
[Termination "Time forfeit"]

1. Nf3 { [%eval 0.19] [%clk 0:05:00] } 1... Nc6 { [%eval 0.42] [%clk 0:05:00] } 2. c4 { [%eval 0.02] [%clk 0:05:07] } 2... e5 { [%eval 0.1] [%clk 0:04:55] } 3. e3 { [%eval -0.04] [%clk 0:05:06] } 3... e4 { [%eval 0.05] [%clk 0:04:57] } 4. Ng1 { [%eval -0.06] [%clk 0:05:02] } 4... Nf6 { [%eval -0.09] [%clk 0:04:51] } 5. f3?! { [%eval -0.68] [%clk 0:05:07] } 5... Bc5?! { [%eval 0.19] [%clk 0:04:43] } 6. fxe4 { [%eval -0.09] [%clk 0:04:47] } 6... Nxe4 { [%eval -0.16] [%clk 0:04:45] } 7. Qg4?! { [%eval -0.85] [%clk 0:04:12] } 1-0


In [27]:
idx = df_audit.sort_values("evals_per_ply_approx").iloc[0]["game_index"]

game = next(
  game for game in eval_games
  if game.game_index == idx
)

print(game.raw_pgn)

[Event "Rated Blitz game"]
[Site "https://lichess.org/Qwv4PuNW"]
[White "ThaJarney"]
[Black "chefbrandolono"]
[Result "1-0"]
[UTCDate "2017.04.30"]
[UTCTime "22:00:24"]
[WhiteElo "1075"]
[BlackElo "1163"]
[WhiteRatingDiff "+43"]
[BlackRatingDiff "-13"]
[ECO "A04"]
[Opening "Zukertort Opening: Dutch Variation"]
[TimeControl "180+0"]
[Termination "Normal"]

1. Nf3 { [%eval 0.18] [%clk 0:03:00] } 1... f5 { [%eval 0.52] [%clk 0:03:00] } 2. g3 { [%eval 0.36] [%clk 0:03:00] } 2... e6 { [%eval 0.51] [%clk 0:02:50] } 3. Bg2 { [%eval 0.52] [%clk 0:03:00] } 3... Bc5 { [%eval 0.72] [%clk 0:02:47] } 4. d4 { [%eval 0.75] [%clk 0:02:56] } 4... Bb6 { [%eval 0.88] [%clk 0:02:45] } 5. d5?! { [%eval 0.24] [%clk 0:02:55] } 5... exd5 { [%eval 0.29] [%clk 0:02:43] } 6. Qxd5 { [%eval 0.29] [%clk 0:02:51] } 6... Ne7 { [%eval 0.47] [%clk 0:02:40] } 7. Qe4?? { [%eval -11.13] [%clk 0:02:46] } 7... d5?? { [%eval 0.04] [%clk 0:02:35] } 8. Qe5 { [%eval -0.34] [%clk 0:02:33] } 8... Qd6? { [%eval 1.76] [%clk 0:02:28

In [28]:
idx = df_audit.sort_values("evals_per_ply_approx").iloc[1]["game_index"]

game = next(
  game for game in eval_games
  if game.game_index == idx
)

print(game.raw_pgn)

[Event "Rated Bullet tournament https://lichess.org/tournament/N4BCgnaA"]
[Site "https://lichess.org/avftQQN9"]
[White "coloviczoki"]
[Black "robrml"]
[Result "1-0"]
[UTCDate "2017.04.30"]
[UTCTime "22:00:01"]
[WhiteElo "2136"]
[BlackElo "2001"]
[WhiteRatingDiff "+8"]
[BlackRatingDiff "-7"]
[ECO "B12"]
[Opening "Caro-Kann Defense: Advance Variation, Botvinnik-Carls Defense"]
[TimeControl "60+0"]
[Termination "Normal"]

1. e4 { [%eval 0.35] [%clk 0:01:00] } 1... c6 { [%eval 0.45] [%clk 0:01:00] } 2. d4 { [%eval 0.46] [%clk 0:00:59] } 2... d5 { [%eval 0.5] [%clk 0:01:00] } 3. e5 { [%eval 0.31] [%clk 0:00:59] } 3... c5 { [%eval 0.3] [%clk 0:01:00] } 4. Nf3 { [%eval 0.1] [%clk 0:00:59] } 4... Nc6 { [%eval 0.39] [%clk 0:00:59] } 5. c3 { [%eval 0.1] [%clk 0:00:58] } 5... Bg4 { [%eval 0.32] [%clk 0:00:59] } 6. dxc5 { [%eval 0.28] [%clk 0:00:56] } 6... e6 { [%eval 0.47] [%clk 0:00:58] } 7. Be3 { [%eval 0.41] [%clk 0:00:55] } 7... Nxe5?? { [%eval 3.45] [%clk 0:00:57] } 8. Bb5+?? { [%eval 0.29] 

In [29]:
print(58/59)
print(70/71)

0.9830508474576272
0.9859154929577465


In [30]:
def print_audit_report(df_audit):
  n_games = len(df_audit)

  print(f"Audited eval games: {n_games}")
  print()

  print("Eval annotations per game:")
  print(df_audit["n_eval_annotations"].describe())
  print()

  print("Clock annotations per game:")
  print(df_audit["n_clock_annotations"].describe())
  print()

  print("Approximate plies per game:")
  print(df_audit["n_plies_approx"].describe())
  print()

  print("Approximate evals per ply:")
  print(df_audit["evals_per_ply_approx"].describe())
  print()

  n_mate = df_audit["has_mate_eval"].sum()
  frac_mate = n_mate / n_games if n_games else 0

  print(f"Games with mate evals: {n_mate} / {n_games}")
  print(f"Fraction with mate evals: {frac_mate:.3f}")

In [31]:
print_audit_report(df_audit)

Audited eval games: 10

Eval annotations per game:
count     10.000000
mean      65.300000
std       40.949969
min       13.000000
25%       38.000000
50%       60.000000
75%       69.250000
max      140.000000
Name: n_eval_annotations, dtype: float64

Clock annotations per game:
count     10.000000
mean      65.500000
std       40.945085
min       13.000000
25%       38.000000
50%       60.500000
75%       70.000000
max      140.000000
Name: n_clock_annotations, dtype: float64

Approximate plies per game:
count     10.000000
mean      65.500000
std       40.945085
min       13.000000
25%       38.000000
50%       60.500000
75%       70.000000
max      140.000000
Name: n_plies_approx, dtype: float64

Approximate evals per ply:
count    10.000000
mean      0.996897
std       0.006577
min       0.983051
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000
Name: evals_per_ply_approx, dtype: float64

Games with mate evals: 4 / 10
Fraction with mate evals: 0.400
